# 07 — Similar News Across TV Channels

**Parte 3 — Multi-video analysis**

Pergunta principal:

> **Is it possible to identify similar news across the two TV channels?**

Este notebook usa os outputs multimodais do notebook de fusão OCR + Speech, especialmente:

```text
outputs_fusion_by_video/<VIDEO_ID>/fusion_multimodal_blocks.csv
```

A unidade de análise é o bloco/notícia estimado (`fusion_multimodal_blocks.csv`).  
O objetivo é comparar blocos RTP vs TVI no mesmo dia e encontrar pares candidatos de notícias semelhantes.

## Definição operacional

Duas notícias são consideradas candidatas a semelhantes quando:

1. pertencem ao mesmo dia;
2. vêm de canais diferentes;
3. têm tema compatível;
4. têm texto OCR/speech semelhante;
5. partilham palavras-chave relevantes.

O resultado é uma lista de **candidate similar news pairs**, não uma verdade absoluta.


## 0. Configuração

A estrutura esperada é:

```text
outputs_fusion_by_video/
├── Telejornal_RTP_Nov_20/
│   └── fusion_multimodal_blocks.csv
├── Telejornal_TVI_Nov_20/
│   └── fusion_multimodal_blocks.csv
...
```


In [ ]:
from pathlib import Path
from collections import Counter
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
BASE_DIR = Path(".")
FUSION_BASE_DIR = BASE_DIR / "outputs_fusion_by_video"
OUTPUT_DIR = BASE_DIR / "outputs_part3_similar_news"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FUSION_FILENAME = "fusion_multimodal_blocks.csv"

# Se quiseres limitar a alguns vídeos:

TARGET_VIDEO_IDS = None

"""
TARGET_VIDEO_IDS = [
    "Telejornal_RTP_Jan_6",
    "Telejornal_TVI_Jan_6",
]
"""

CHANNEL_A = "RTP"
CHANNEL_B = "TVI"

# ------------------------------------------------------------
# Peso dos textos
# ------------------------------------------------------------
# OCR lower-third tende a conter título/subtítulo, por isso pesa mais.
OCR_TEXT_WEIGHT = 3
SPEECH_TEXT_WEIGHT = 1
THEME_TEXT_WEIGHT = 2

# ------------------------------------------------------------
# Thresholds de matching
# ------------------------------------------------------------
STRONG_TEXT_SIM_THRESHOLD = 0.45
POSSIBLE_TEXT_SIM_THRESHOLD = 0.30

MIN_SHARED_KEYWORDS_STRONG = 2
MIN_SHARED_KEYWORDS_POSSIBLE = 2

KEEP_MATCH_LABELS = ["strong_match", "possible_match"]

# Já fica preparado para a pergunta 9.
SAME_TIME_WINDOW_MINUTES = 15

print("FUSION_BASE_DIR:", FUSION_BASE_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


## 1. Funções auxiliares

Inclui parsing de `VIDEO_ID`, limpeza/tokenização em português, compatibilidade de temas e keywords partilhadas.


In [ ]:
def seconds_to_hhmmss(seconds):
    if pd.isna(seconds):
        return ""
    seconds = int(round(float(seconds)))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


def parse_video_id(video_id):
    """
    Esperado:
    Telejornal_RTP_Nov_20
    Telejornal_TVI_Dec_2
    """
    parts = str(video_id).split("_")
    if len(parts) >= 4 and parts[0].lower().startswith("telejornal"):
        channel = parts[1]
        date_label = "_".join(parts[2:])
    elif len(parts) >= 3:
        channel = parts[1]
        date_label = "_".join(parts[2:])
    else:
        channel = "unknown"
        date_label = video_id

    return {"video_id": video_id, "channel": channel, "date_label": date_label}


STOPWORDS_PT = {
    "de","a","o","e","que","do","da","em","um","uma","para","com","não","os","as","no","na","por",
    "se","ao","dos","das","mais","como","é","foi","são","ser","tem","também","ou","à","às","nos",
    "nas","sobre","entre","até","sem","já","lhe","ele","ela","eles","elas","sua","seu","suas","seus",
    "este","esta","estes","estas","isso","isto","há","vai","ter","mas","muito","muita","muitos","muitas",
    "porque","quando","onde","quem","qual","quais","todo","toda","todos","todas","num","numa","pelo","pela",
    "pelos","pelas","aos","ainda","só","era","foram","será","serem","nosso","nossa","seja","forma",
    "me", "te", "vos", "nos", "sido", "tinha", "tinham", "pode", "podem", "deve", "devem", "estar", "está",
    "estão", "faz", "fazer", "fez", "ano", "anos", "dia", "dias", "hoje", "ontem", "amanhã",
    "and", "the", "for", "with", "from", "this", "that", "are", "was", "were",
    "you", "your", "our", "their", "his", "her", "not", "but", "has", "have",
    "had", "will", "would", "can", "could", "should"
}

STRUCTURAL_WORDS = {
    "telejornal", "jornal", "nacional", "direto", "directo", "tvi", "rtp", "sic", "cnn", "cmtv",
    "notícias", "noticia", "noticias", "edição", "edicao", "especial", "última", "ultima", "hora", "minuto",
    "portugal", "portuguesa", "português", "portugues", "portugueses", "www", "pt", "hd",
    "noticiasrtp", "rtppt", "notíciasrtp", "qui", "seg", "ter", "qua", "sex", "sab", "dom",
    "imagem", "arquivo", "fonte", "agência", "agencia"
}


def clean_text_pt(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower()
    text = text.replace("\\n", " ")
    text = re.sub(r"[^a-záàâãéèêíóôõúç0-9\s]", " ", text)
    text = re.sub(r"\b(b|br)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_pt(text, remove_stopwords=True, remove_structural=True, min_len=3):
    tokens = clean_text_pt(text).split()
    tokens = [t for t in tokens if len(t) >= min_len]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS_PT]

    if remove_structural:
        tokens = [t for t in tokens if t not in STRUCTURAL_WORDS]

    return tokens


def split_items(value):
    if pd.isna(value) or str(value).strip() == "":
        return set()
    return {item.strip() for item in str(value).split(",") if item.strip()}


def concat_non_empty(values, sep=" "):
    out = []
    for v in values:
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            out.append(s)
    return sep.join(out)


RELATED_THEME_GROUPS = {
    "politics": {"Eleições/Campanha", "Governo/Partidos", "Sondagens"},
    "public_services": {"Saúde", "Educação", "Habitação"},
    "economy_work": {"Economia", "Greves/Trabalho"},
    "security_justice": {"Justiça/Segurança", "Incêndios/Proteção Civil"},
    "international": {"Internacional"},
    "sports": {"Desporto"},
    "culture": {"Cultura"},
    "environment_weather": {"Ambiente", "Ambiente/Clima", "Meteorologia", "Incêndios/Proteção Civil"},
    "transport": {"Transportes", "Transportes/Mobilidade"},
}


def theme_group(theme):
    if pd.isna(theme) or str(theme).strip() in ["", "Other/Unknown", "No speech"]:
        return "unknown"
    theme = str(theme).strip()
    for group_name, themes in RELATED_THEME_GROUPS.items():
        if theme in themes:
            return group_name
    return theme


def themes_compatible(row_a, row_b):
    theme_a = str(row_a.get("final_theme", "Other/Unknown"))
    theme_b = str(row_b.get("final_theme", "Other/Unknown"))

    if theme_a != "Other/Unknown" and theme_a == theme_b:
        return True

    if theme_group(theme_a) != "unknown" and theme_group(theme_a) == theme_group(theme_b):
        return True

    themes_a = set()
    themes_b = set()

    for col in ["ocr_themes_present", "speech_themes_present"]:
        themes_a |= split_items(row_a.get(col, ""))
        themes_b |= split_items(row_b.get(col, ""))

    return len(themes_a & themes_b) > 0

def main_themes_compatible(row_a, row_b):
    theme_a = str(row_a.get("final_theme", "Other/Unknown")).strip()
    theme_b = str(row_b.get("final_theme", "Other/Unknown")).strip()

    if theme_a in ["", "Other/Unknown", "No speech"]:
        return False
    if theme_b in ["", "Other/Unknown", "No speech"]:
        return False

    if theme_a == theme_b:
        return True

    if theme_group(theme_a) != "unknown" and theme_group(theme_a) == theme_group(theme_b):
        return True

    return False


def top_keywords(text, max_keywords=12):
    tokens = tokenize_pt(text)
    counts = Counter(tokens)
    return [w for w, _ in counts.most_common(max_keywords)]


def shared_keywords(text_a, text_b, max_keywords=12):
    kw_a = set(top_keywords(text_a, max_keywords=max_keywords))
    kw_b = set(top_keywords(text_b, max_keywords=max_keywords))
    return sorted(kw_a & kw_b)


## 2. Carregar blocos multimodais de todos os telejornais

Cada linha do `fusion_multimodal_blocks.csv` é tratada como um **estimated multimodal news block**.


In [ ]:
def find_fusion_files():
    if TARGET_VIDEO_IDS is not None:
        files = []
        for video_id in TARGET_VIDEO_IDS:
            path = FUSION_BASE_DIR / video_id / FUSION_FILENAME
            if path.exists():
                files.append(path)
            else:
                print("Missing:", path)
        return files

    if not FUSION_BASE_DIR.exists():
        raise FileNotFoundError(
            f"FUSION_BASE_DIR não existe: {FUSION_BASE_DIR.resolve()}\n"
            "Corre primeiro o notebook 06_ocr_speech_fusion para vários vídeos."
        )

    return sorted(FUSION_BASE_DIR.glob(f"*/{FUSION_FILENAME}"))


fusion_files = find_fusion_files()

print("Fusion files found:", len(fusion_files))
for f in fusion_files:
    print(" -", f.parent.name, "->", f.name)

if len(fusion_files) == 0:
    raise FileNotFoundError(
        "Não encontrei fusion_multimodal_blocks.csv em outputs_fusion_by_video/*."
    )

all_blocks = []

for file_path in fusion_files:
    video_id = file_path.parent.name
    meta = parse_video_id(video_id)

    df = pd.read_csv(file_path)
    df["video_id"] = video_id
    df["channel"] = meta["channel"]
    df["date_label"] = meta["date_label"]
    df["source_file"] = str(file_path)

    all_blocks.append(df)

blocks = pd.concat(all_blocks, ignore_index=True)

print("All blocks shape:", blocks.shape)
display(blocks.head())
print("Channels:", blocks["channel"].value_counts(dropna=False).to_dict())
print("Dates:", sorted(blocks["date_label"].dropna().unique().tolist()))


## 3. Normalizar colunas e criar texto representativo por bloco

Não usamos só `final_theme`. Para aproximar uma notícia específica, criamos `block_text` com:

- tema final;
- temas OCR/speech presentes;
- texto OCR do lower-third;
- excerto speech;
- candidatos/entidades mencionados.

O OCR recebe mais peso porque costuma conter o título/subtítulo da notícia.


In [ ]:
required_time_cols = ["start_sec", "end_sec"]
missing = [c for c in required_time_cols if c not in blocks.columns]
if missing:
    raise ValueError(f"Faltam colunas temporais nos blocos: {missing}. Colunas disponíveis: {list(blocks.columns)}")

blocks_norm = blocks.copy()

blocks_norm["block_id"] = pd.to_numeric(blocks_norm.get("block_id"), errors="coerce")
blocks_norm["start_sec"] = pd.to_numeric(blocks_norm["start_sec"], errors="coerce")
blocks_norm["end_sec"] = pd.to_numeric(blocks_norm["end_sec"], errors="coerce")
blocks_norm = blocks_norm.dropna(subset=["start_sec", "end_sec"]).copy()

blocks_norm["duration_sec"] = blocks_norm["end_sec"] - blocks_norm["start_sec"]
blocks_norm = blocks_norm[blocks_norm["duration_sec"] > 0].copy()

if "start_time" not in blocks_norm.columns:
    blocks_norm["start_time"] = blocks_norm["start_sec"].apply(seconds_to_hhmmss)
if "end_time" not in blocks_norm.columns:
    blocks_norm["end_time"] = blocks_norm["end_sec"].apply(seconds_to_hhmmss)
if "duration_min" not in blocks_norm.columns:
    blocks_norm["duration_min"] = blocks_norm["duration_sec"] / 60

for col in [
    "final_theme", "ocr_theme", "speech_theme",
    "ocr_themes_present", "speech_themes_present",
    "ocr_candidates_present", "speech_candidates_present",
    "ocr_text", "speech_excerpt", "agreement", "needs_review"
]:
    if col not in blocks_norm.columns:
        blocks_norm[col] = ""

blocks_norm["final_theme"] = blocks_norm["final_theme"].replace("", np.nan).fillna(
    blocks_norm["ocr_theme"].replace("", np.nan)
).fillna("Other/Unknown")

UNKNOWN_THEMES = {"Other/Unknown", "No speech", ""}

n_before = len(blocks_norm)

blocks_norm = blocks_norm[
    blocks_norm["final_theme"].notna()
    & ~blocks_norm["final_theme"].astype(str).str.strip().isin(UNKNOWN_THEMES)
].copy()

print("Removed unknown-theme blocks:", n_before - len(blocks_norm))
print("Remaining blocks:", len(blocks_norm))

blocks_norm["block_key"] = (
    blocks_norm["video_id"].astype(str)
    + "::"
    + blocks_norm["block_id"].astype(str)
)

def build_block_text(row):
    theme_text = concat_non_empty([
        row.get("final_theme", ""),
        row.get("ocr_theme", ""),
        row.get("speech_theme", ""),
        row.get("ocr_themes_present", ""),
        row.get("speech_themes_present", ""),
        row.get("ocr_candidates_present", ""),
        row.get("speech_candidates_present", ""),
    ])

    ocr_text = str(row.get("ocr_text", "") or "")
    speech_text = str(row.get("speech_excerpt", "") or "")

    parts = []
    parts.extend([theme_text] * THEME_TEXT_WEIGHT)
    parts.extend([ocr_text] * OCR_TEXT_WEIGHT)
    parts.extend([speech_text] * SPEECH_TEXT_WEIGHT)

    return clean_text_pt(" ".join(parts))

blocks_norm["block_text"] = blocks_norm.apply(build_block_text, axis=1)
blocks_norm["block_text_len"] = blocks_norm["block_text"].str.len()
blocks_norm["keywords"] = blocks_norm["block_text"].apply(lambda x: ", ".join(top_keywords(x, max_keywords=10)))

blocks_norm = blocks_norm[blocks_norm["block_text_len"] > 0].copy()

display_cols = [
    "video_id", "channel", "date_label", "block_id", "start_time", "end_time",
    "duration_min", "final_theme", "ocr_text", "speech_excerpt", "keywords"
]
display_cols = [c for c in display_cols if c in blocks_norm.columns]

display(blocks_norm[display_cols].head(20))

blocks_norm.to_csv(OUTPUT_DIR / "similar_news_blocks_input_normalized.csv", index=False)


## 4. Cobertura por canal e data

Verificamos se existem blocos dos dois canais nas mesmas datas.


In [ ]:
coverage = (
    blocks_norm
    .groupby(["date_label", "channel"])
    .agg(
        n_blocks=("block_key", "nunique"),
        total_duration_min=("duration_min", "sum"),
        n_themes=("final_theme", "nunique"),
    )
    .reset_index()
    .sort_values(["date_label", "channel"])
)

display(coverage)

same_day_channel_counts = (
    coverage
    .pivot_table(index="date_label", columns="channel", values="n_blocks", aggfunc="sum")
    .fillna(0)
    .reset_index()
)

display(same_day_channel_counts)

valid_dates = []
for date_label, group in blocks_norm.groupby("date_label"):
    channels = set(group["channel"].dropna().astype(str))
    if CHANNEL_A in channels and CHANNEL_B in channels:
        valid_dates.append(date_label)

print("Dates with both channels:", valid_dates)

if not valid_dates:
    raise ValueError(
        f"Não há nenhuma data com ambos os canais {CHANNEL_A} e {CHANNEL_B}."
    )

coverage.to_csv(OUTPUT_DIR / "similar_news_channel_date_coverage.csv", index=False)


## 5. Calcular similaridade textual entre blocos RTP e TVI

Para cada dia com os dois canais, comparamos todos os blocos RTP com todos os blocos TVI.

A métrica principal é **TF-IDF + cosine similarity** sobre `block_text`.


In [ ]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except Exception as e:
    SKLEARN_AVAILABLE = False
    print("sklearn não disponível:", e)

if not SKLEARN_AVAILABLE:
    raise ImportError("Este notebook precisa de sklearn para calcular TF-IDF + cosine similarity.")

vectorizer = TfidfVectorizer(
    tokenizer=lambda x: tokenize_pt(x),
    token_pattern=None,
    lowercase=False,
    min_df=1,
    ngram_range=(1, 2),
)

X = vectorizer.fit_transform(blocks_norm["block_text"].fillna("").astype(str).tolist())

block_key_to_idx = {
    key: idx
    for idx, key in enumerate(blocks_norm["block_key"].tolist())
}

pair_rows = []

for date_label in valid_dates:
    day = blocks_norm[blocks_norm["date_label"] == date_label].copy()

    a_blocks = day[day["channel"] == CHANNEL_A].copy()
    b_blocks = day[day["channel"] == CHANNEL_B].copy()

    for _, a in a_blocks.iterrows():
        for _, b in b_blocks.iterrows():
            idx_a = block_key_to_idx[a["block_key"]]
            idx_b = block_key_to_idx[b["block_key"]]

            sim = float(cosine_similarity(X[idx_a], X[idx_b])[0, 0])
            # compatible_theme = themes_compatible(a, b) # teste q esta me a dar alguns erros, ver isto mais tarde n esquecerr poh
            compatible_theme = main_themes_compatible(a, b)
            
            shared_kw = shared_keywords(
                str(a.get("ocr_text", "")),
                str(b.get("ocr_text", "")),
                max_keywords=15
            )

            KEYWORD_SCORE_CAP = 4
            keyword_score = min(len(shared_kw), KEYWORD_SCORE_CAP) / KEYWORD_SCORE_CAP
            theme_score = 1.0 if compatible_theme else 0.0

            combined_score = (
                0.25 * sim
                + 0.35 * theme_score
                + 0.40 * keyword_score
            )

            time_diff_min = abs(float(a["start_sec"]) - float(b["start_sec"])) / 60

            pair_rows.append({
                "date_label": date_label,

                f"{CHANNEL_A.lower()}_video_id": a["video_id"],
                f"{CHANNEL_B.lower()}_video_id": b["video_id"],

                f"{CHANNEL_A.lower()}_block_id": int(a["block_id"]) if pd.notna(a["block_id"]) else np.nan,
                f"{CHANNEL_B.lower()}_block_id": int(b["block_id"]) if pd.notna(b["block_id"]) else np.nan,

                f"{CHANNEL_A.lower()}_start_sec": float(a["start_sec"]),
                f"{CHANNEL_B.lower()}_start_sec": float(b["start_sec"]),
                f"{CHANNEL_A.lower()}_start_time": a["start_time"],
                f"{CHANNEL_B.lower()}_start_time": b["start_time"],
                f"{CHANNEL_A.lower()}_end_time": a["end_time"],
                f"{CHANNEL_B.lower()}_end_time": b["end_time"],

                f"{CHANNEL_A.lower()}_duration_min": a.get("duration_min", np.nan),
                f"{CHANNEL_B.lower()}_duration_min": b.get("duration_min", np.nan),

                f"{CHANNEL_A.lower()}_theme": a.get("final_theme", ""),
                f"{CHANNEL_B.lower()}_theme": b.get("final_theme", ""),

                f"{CHANNEL_A.lower()}_ocr_text": a.get("ocr_text", ""),
                f"{CHANNEL_B.lower()}_ocr_text": b.get("ocr_text", ""),
                f"{CHANNEL_A.lower()}_speech_excerpt": a.get("speech_excerpt", ""),
                f"{CHANNEL_B.lower()}_speech_excerpt": b.get("speech_excerpt", ""),

                "combined_score": combined_score,
                "text_similarity": sim,
                "theme_compatible": compatible_theme,
                "shared_keywords": ", ".join(shared_kw),
                "n_shared_keywords": len(shared_kw),
                "keyword_score": keyword_score,

                "time_diff_min": time_diff_min,
                "same_time_window_15min": time_diff_min <= SAME_TIME_WINDOW_MINUTES,
            })

all_pairs = pd.DataFrame(pair_rows)

print("All RTP/TVI same-day pairs:", len(all_pairs))
display(all_pairs.sort_values("combined_score", ascending=False).head(20))

all_pairs.to_csv(OUTPUT_DIR / "similar_news_all_pairs.csv", index=False)


## 6. Classificar candidatos

A classificação combina similaridade textual, compatibilidade temática e keywords comuns.

Esta é a parte que responde diretamente à pergunta:

> Is it possible to identify similar news across the two TV channels?


In [ ]:
# ============================================================
# Classificar matches usando sobretudo o combined_score
# ============================================================

COMBINED_STRONG_THRESHOLD = 0.60
COMBINED_POSSIBLE_THRESHOLD = 0.45

def classify_match(row):
    combined = float(row["combined_score"])
    sim = float(row["text_similarity"])
    theme_ok = bool(row["theme_compatible"])
    n_kw = int(row["n_shared_keywords"])

    # Proteção mínima para não aceitar pares sem nenhuma evidência real
    has_minimum_evidence = (
        theme_ok or       # temas compatíveis
        n_kw >= 2 or      # pelo menos 2 keywords partilhadas
        sim >= 0.20       # texto suficientemente parecido
    )

    if not has_minimum_evidence:
        return "weak_or_no_match"

    if combined >= COMBINED_STRONG_THRESHOLD:
        return "strong_match"

    if combined >= COMBINED_POSSIBLE_THRESHOLD:
        return "possible_match"

    return "weak_or_no_match"


all_pairs["match_label"] = all_pairs.apply(classify_match, axis=1)

candidate_pairs = (
    all_pairs[all_pairs["match_label"].isin(KEEP_MATCH_LABELS)]
    .copy()
    .sort_values(["date_label", "combined_score"], ascending=[True, False])
    .reset_index(drop=True)
)

print("Combined strong threshold:", COMBINED_STRONG_THRESHOLD)
print("Combined possible threshold:", COMBINED_POSSIBLE_THRESHOLD)
print("Candidate pairs:", len(candidate_pairs))

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", None,
    "display.max_colwidth", 300,
    "display.width", 3000
):
    display(candidate_pairs.head(100))

match_summary = (
    all_pairs["match_label"]
    .value_counts()
    .reset_index()
)

match_summary.columns = ["match_label", "n_pairs"]
match_summary["percentage"] = (
    match_summary["n_pairs"] / max(len(all_pairs), 1) * 100
).round(2)

display(match_summary)

candidate_pairs.to_csv(
    OUTPUT_DIR / "similar_news_candidate_pairs.csv",
    index=False
)

match_summary.to_csv(
    OUTPUT_DIR / "similar_news_match_label_summary.csv",
    index=False
)

In [ ]:
display(
    all_pairs["match_label"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "match_label", "match_label": "n_pairs"})
)

display(
    all_pairs[
        [
            "date_label",
            "match_label",
            "combined_score",
            "text_similarity",
            "theme_compatible",
            "n_shared_keywords",
            "shared_keywords",
            f"{CHANNEL_A.lower()}_block_id",
            f"{CHANNEL_B.lower()}_block_id",
        ]
    ]
    .sort_values(["match_label", "combined_score"], ascending=[True, False])
    .head(50)
)

## 7. Greedy one-to-one matching

Um bloco RTP pode parecer semelhante a vários blocos TVI.  
Para uma tabela mais limpa de apresentação, criamos uma versão **one-to-one**:

1. ordenar candidatos por score;
2. escolher o melhor par;
3. impedir que o mesmo bloco seja usado novamente.


In [ ]:
unique_rows = []

for date_label, group in candidate_pairs.groupby("date_label"):
    group = group.sort_values("combined_score", ascending=False).copy()

    used_a = set()
    used_b = set()

    for _, row in group.iterrows():
        a_key = (row[f"{CHANNEL_A.lower()}_video_id"], row[f"{CHANNEL_A.lower()}_block_id"])
        b_key = (row[f"{CHANNEL_B.lower()}_video_id"], row[f"{CHANNEL_B.lower()}_block_id"])

        if a_key in used_a or b_key in used_b:
            continue

        unique_rows.append(row.to_dict())
        used_a.add(a_key)
        used_b.add(b_key)

unique_matches = pd.DataFrame(unique_rows)

if not unique_matches.empty:
    unique_matches = unique_matches.sort_values(["date_label", "combined_score"], ascending=[True, False]).reset_index(drop=True)

print("Unique candidate matches:", len(unique_matches))
display(unique_matches.head(20))

unique_matches.to_csv(OUTPUT_DIR / "similar_news_greedy_unique_matches.csv", index=False)


## 8. Visualizações e resumo por data


In [ ]:
if not all_pairs.empty:
    plt.figure(figsize=(10, 4))
    plt.hist(all_pairs["text_similarity"].dropna(), bins=30)
    plt.title("Distribution of RTP vs TVI block text similarity")
    plt.xlabel("TF-IDF cosine similarity")
    plt.ylabel("n_pairs")
    plt.tight_layout()
    plt.show()

if not candidate_pairs.empty:
    candidate_by_date = (
        candidate_pairs
        .groupby(["date_label", "match_label"])
        .size()
        .reset_index(name="n_candidate_pairs")
    )

    display(candidate_by_date)

    plt.figure(figsize=(10, 4))
    totals = candidate_pairs.groupby("date_label").size().sort_index()
    plt.bar(totals.index.astype(str), totals.values)
    plt.title("Candidate similar news pairs by date")
    plt.xlabel("date_label")
    plt.ylabel("n_candidate_pairs")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    candidate_by_date.to_csv(OUTPUT_DIR / "similar_news_candidate_pairs_by_date.csv", index=False)
else:
    print("No candidate pairs found with the current thresholds.")

if not unique_matches.empty:
    plt.figure(figsize=(10, 4))
    top = unique_matches.sort_values("combined_score", ascending=False).head(20).copy()
    labels = [
        f"{r['date_label']} | {int(r[f'{CHANNEL_A.lower()}_block_id'])}-{int(r[f'{CHANNEL_B.lower()}_block_id'])}"
        for _, r in top.iterrows()
    ]
    plt.bar(range(len(top)), top["combined_score"])
    plt.title("Top unique similar-news matches")
    plt.xlabel("match")
    plt.ylabel("combined_score")
    plt.xticks(range(len(top)), labels, rotation=90)
    plt.tight_layout()
    plt.show()


## 9. Tabela de validação manual

Esta é a tabela principal para validar qualitativamente os pares encontrados.


In [ ]:
validation_cols = [
    "date_label",
    "match_label",
    "combined_score",
    "text_similarity",
    "theme_compatible",
    "shared_keywords",
    "time_diff_min",
    "same_time_window_15min",

    f"{CHANNEL_A.lower()}_block_id",
    f"{CHANNEL_A.lower()}_start_time",
    f"{CHANNEL_A.lower()}_theme",
    f"{CHANNEL_A.lower()}_ocr_text",
    f"{CHANNEL_A.lower()}_speech_excerpt",

    f"{CHANNEL_B.lower()}_block_id",
    f"{CHANNEL_B.lower()}_start_time",
    f"{CHANNEL_B.lower()}_theme",
    f"{CHANNEL_B.lower()}_ocr_text",
    f"{CHANNEL_B.lower()}_speech_excerpt",
]

validation_cols = [c for c in validation_cols if c in unique_matches.columns]

manual_validation_table = unique_matches[validation_cols].copy() if not unique_matches.empty else pd.DataFrame(columns=validation_cols)

if not manual_validation_table.empty:
    manual_validation_table["manual_check"] = ""
    manual_validation_table["interpretation"] = manual_validation_table["match_label"].map({
        "strong_match": "Likely same news/event",
        "possible_match": "Possible similar news/event",
    }).fillna("Needs review")

display(manual_validation_table.head(50))

manual_validation_table.to_csv(OUTPUT_DIR / "similar_news_manual_validation_table.csv", index=False)


## 10. Cobertura temática por canal

Esta parte é complementar.

Mesmo quando não conseguimos garantir a mesma notícia específica, podemos ver se os canais cobrem temas semelhantes.


In [ ]:
theme_coverage = (
    blocks_norm
    .groupby(["date_label", "channel", "final_theme"])
    .agg(
        n_blocks=("block_key", "nunique"),
        total_duration_min=("duration_min", "sum"),
    )
    .reset_index()
    .sort_values(["date_label", "channel", "n_blocks"], ascending=[True, True, False])
)

display(theme_coverage.head(40))

theme_coverage.to_csv(OUTPUT_DIR / "similar_news_theme_coverage_by_channel.csv", index=False)

theme_overlap_rows = []

for date_label in valid_dates:
    day = theme_coverage[theme_coverage["date_label"] == date_label]

    themes_a = set(day[day["channel"] == CHANNEL_A]["final_theme"])
    themes_b = set(day[day["channel"] == CHANNEL_B]["final_theme"])

    shared = sorted((themes_a & themes_b) - {"Other/Unknown", ""})
    only_a = sorted((themes_a - themes_b) - {"Other/Unknown", ""})
    only_b = sorted((themes_b - themes_a) - {"Other/Unknown", ""})

    theme_overlap_rows.append({
        "date_label": date_label,
        "shared_themes": ", ".join(shared),
        f"only_{CHANNEL_A.lower()}_themes": ", ".join(only_a),
        f"only_{CHANNEL_B.lower()}_themes": ", ".join(only_b),
        "n_shared_themes": len(shared),
        f"n_only_{CHANNEL_A.lower()}": len(only_a),
        f"n_only_{CHANNEL_B.lower()}": len(only_b),
    })

theme_overlap = pd.DataFrame(theme_overlap_rows)

display(theme_overlap)

theme_overlap.to_csv(OUTPUT_DIR / "similar_news_theme_overlap_by_date.csv", index=False)


## 11. Síntese automática para relatório/apresentação


In [ ]:
n_total_pairs = len(all_pairs)
n_candidate_pairs = len(candidate_pairs)
n_unique_matches = len(unique_matches)
n_strong_unique = int((unique_matches["match_label"] == "strong_match").sum()) if not unique_matches.empty else 0
n_possible_unique = int((unique_matches["match_label"] == "possible_match").sum()) if not unique_matches.empty else 0

dates_analysed = ", ".join(map(str, valid_dates))

top_examples_text = ""

if not manual_validation_table.empty:
    example_rows = []
    for _, r in manual_validation_table.head(5).iterrows():
        example_rows.append(
            f"- {r['date_label']} | {CHANNEL_A} block {r[f'{CHANNEL_A.lower()}_block_id']} "
            f"({r[f'{CHANNEL_A.lower()}_start_time']}) ↔ "
            f"{CHANNEL_B} block {r[f'{CHANNEL_B.lower()}_block_id']} "
            f"({r[f'{CHANNEL_B.lower()}_start_time']}), "
            f"label={r['match_label']}, similarity={r['text_similarity']:.2f}, "
            f"shared keywords={r['shared_keywords']}"
        )
    top_examples_text = "\n".join(example_rows)

summary_text = f"""
### Question 8 — Similar news across TV channels

We used the multimodal OCR+Speech news blocks as the unit of comparison.
Each block was represented by a combined text containing the final topic, OCR lower-third text, speech excerpt, and detected themes/entities.

To identify candidate similar news across channels, we compared {CHANNEL_A} and {CHANNEL_B} blocks from the same date using TF-IDF cosine similarity.
A candidate pair was considered stronger when it had compatible topics, higher textual similarity and shared relevant keywords.

- Dates analysed: {dates_analysed}
- Total same-day cross-channel block pairs: {n_total_pairs}
- Candidate similar-news pairs: {n_candidate_pairs}
- Unique one-to-one candidate matches: {n_unique_matches}
- Strong unique matches: {n_strong_unique}
- Possible unique matches: {n_possible_unique}

Interpretation:
The method does not claim perfect story identification.
Instead, it produces candidate pairs of similar news/events that can be manually validated.
This is appropriate because the underlying blocks are estimated OCR/Speech-based news segments.

Top candidate examples:
{top_examples_text}
"""

display(Markdown(summary_text))

with open(OUTPUT_DIR / "question8_similar_news_summary.md", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("Saved summary to:", OUTPUT_DIR / "question8_similar_news_summary.md")


## 12. Final evidence for Question 8

In [ ]:
# Final summary metrics for Question 8

summary_q8 = pd.DataFrame({
    "metric": [
        "Blocks analysed after filtering",
        "All same-day RTP/TVI pairs",
        "Candidate similar-news pairs",
        "Unique final matches",
    ],
    "value": [
        len(blocks_norm),
        len(all_pairs),
        len(candidate_pairs),
        len(unique_matches),
    ]
})

display(summary_q8)

summary_q8.to_csv(OUTPUT_DIR / "q8_final_summary_metrics.csv", index=False)

In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(summary_q8["metric"], summary_q8["value"])

plt.title("Similar news detection pipeline")
plt.ylabel("Number of items")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
if not unique_matches.empty:
    top_matches = unique_matches.sort_values("combined_score", ascending=False).head(15).copy()

    labels = [
        f"{r['date_label']} | {CHANNEL_A} {int(r[f'{CHANNEL_A.lower()}_block_id'])} - {CHANNEL_B} {int(r[f'{CHANNEL_B.lower()}_block_id'])}"
        for _, r in top_matches.iterrows()
    ]

    plt.figure(figsize=(10, 5))
    plt.bar(labels, top_matches["combined_score"])
    plt.title("Top final similar-news matches")
    plt.ylabel("Combined similarity score")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()
else:
    print("No unique matches available.")

In [ ]:
if not unique_matches.empty:
    theme_col = f"{CHANNEL_A.lower()}_theme"

    matches_by_theme = (
        unique_matches
        .groupby(theme_col)
        .size()
        .reset_index(name="n_matches")
        .sort_values("n_matches", ascending=False)
    )

    display(matches_by_theme)

    plt.figure(figsize=(9, 4))
    plt.bar(matches_by_theme[theme_col], matches_by_theme["n_matches"])
    plt.title("Final similar-news matches by theme")
    plt.ylabel("Number of final matches")
    plt.xlabel("Theme")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    matches_by_theme.to_csv(OUTPUT_DIR / "q8_final_matches_by_theme.csv", index=False)
else:
    print("No unique matches available.")

In [ ]:
presentation_cols = [
    "date_label",
    "match_label",
    "combined_score",
    "text_similarity",
    "shared_keywords",
    "time_diff_min",
    "same_time_window_15min",

    f"{CHANNEL_A.lower()}_block_id",
    f"{CHANNEL_A.lower()}_start_time",
    f"{CHANNEL_A.lower()}_theme",
    f"{CHANNEL_A.lower()}_ocr_text",

    f"{CHANNEL_B.lower()}_block_id",
    f"{CHANNEL_B.lower()}_start_time",
    f"{CHANNEL_B.lower()}_theme",
    f"{CHANNEL_B.lower()}_ocr_text",
]

presentation_cols = [c for c in presentation_cols if c in unique_matches.columns]

q8_presentation_table = (
    unique_matches[presentation_cols]
    .sort_values("combined_score", ascending=False)
    .head(20)
    .copy()
)

display(q8_presentation_table)

q8_presentation_table.to_csv(
    OUTPUT_DIR / "q8_top_matches_for_presentation.csv",
    index=False
)

## 13. Question 9 — Do similar news appear in the same time window?

In [ ]:
def classify_time_window(diff_min):
    if pd.isna(diff_min):
        return "unknown"
    if diff_min <= 5:
        return "very_close_<=5min"
    if diff_min <= 15:
        return "same_window_5_15min"
    return "different_window_>15min"


q9_matches = unique_matches.copy()

q9_matches["time_window_label"] = q9_matches["time_diff_min"].apply(classify_time_window)

# Como os frames são 1 FPS:
# start_sec ≈ frame
# Se os teus ficheiros de frames começarem em 1 em vez de 0, muda para 1.
APPROX_FRAME_OFFSET = 0

q9_matches[f"{CHANNEL_A.lower()}_start_frame"] = (
    q9_matches[f"{CHANNEL_A.lower()}_start_sec"]
    .round()
    .astype("Int64")
    + APPROX_FRAME_OFFSET
)

q9_matches[f"{CHANNEL_B.lower()}_start_frame"] = (
    q9_matches[f"{CHANNEL_B.lower()}_start_sec"]
    .round()
    .astype("Int64")
    + APPROX_FRAME_OFFSET
)

display(q9_matches[[
    "date_label",
    "match_label",
    "combined_score",
    "time_diff_min",
    "time_window_label",

    f"{CHANNEL_A.lower()}_block_id",
    f"{CHANNEL_A.lower()}_start_frame",
    f"{CHANNEL_A.lower()}_theme",

    f"{CHANNEL_B.lower()}_block_id",
    f"{CHANNEL_B.lower()}_start_frame",
    f"{CHANNEL_B.lower()}_theme",

    "shared_keywords"
]].head(30))

In [ ]:
q9_summary = (
    q9_matches["time_window_label"]
    .value_counts()
    .reset_index()
)

q9_summary.columns = ["time_window_label", "n_matches"]
q9_summary["percentage"] = (
    q9_summary["n_matches"] / max(len(q9_matches), 1) * 100
).round(2)

display(q9_summary)

q9_summary.to_csv(
    OUTPUT_DIR / "q9_time_window_summary.csv",
    index=False
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(q9_summary["time_window_label"], q9_summary["n_matches"])
plt.title("Do similar news appear in the same time window?")
plt.ylabel("Number of final matches")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(
    q9_matches[f"{CHANNEL_A.lower()}_start_sec"] / 60,
    q9_matches[f"{CHANNEL_B.lower()}_start_sec"] / 60
)

max_minute = max(
    q9_matches[f"{CHANNEL_A.lower()}_start_sec"].max(),
    q9_matches[f"{CHANNEL_B.lower()}_start_sec"].max()
) / 60

plt.plot([0, max_minute], [0, max_minute], linestyle="--")

plt.title("Start time of similar news: RTP vs TVI")
plt.xlabel(f"{CHANNEL_A} start minute")
plt.ylabel(f"{CHANNEL_B} start minute")
plt.tight_layout()
plt.show()

In [ ]:
top_timeline = (
    q9_matches
    .sort_values("combined_score", ascending=False)
    .head(20)
    .copy()
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 5))

for i, row in top_timeline.iterrows():
    rtp_min = row[f"{CHANNEL_A.lower()}_start_sec"] / 60
    tvi_min = row[f"{CHANNEL_B.lower()}_start_sec"] / 60

    # linha entre os dois tempos
    plt.plot([rtp_min, tvi_min], [i, i], marker="o")

plt.yticks(
    range(len(top_timeline)),
    [
        f"{row['date_label']} | {CHANNEL_A} {int(row[f'{CHANNEL_A.lower()}_block_id'])} - {CHANNEL_B} {int(row[f'{CHANNEL_B.lower()}_block_id'])}"
        for _, row in top_timeline.iterrows()
    ]
)

plt.xlabel("Start minute in the newscast")
plt.title("Temporal position of top similar-news matches")
plt.tight_layout()
plt.show()

## 13. Outputs gerados

Ficheiros principais em `outputs_part3_similar_news/`:

```text
similar_news_blocks_input_normalized.csv
similar_news_channel_date_coverage.csv
similar_news_all_pairs.csv
similar_news_candidate_pairs.csv
similar_news_greedy_unique_matches.csv
similar_news_manual_validation_table.csv
similar_news_theme_coverage_by_channel.csv
similar_news_theme_overlap_by_date.csv
question8_similar_news_summary.md
```

Para responder à pergunta 8, os mais importantes são:

```text
similar_news_greedy_unique_matches.csv
similar_news_manual_validation_table.csv
question8_similar_news_summary.md
```

A coluna `time_diff_min` já fica preparada para a pergunta 9.
